In [3]:
from kitefs import FeatureStore



In [ ]:
# !kitefs apply

Applied feature groups: listing_features, town_market_features.


In [4]:
store = FeatureStore()


In [4]:
result = store.apply()
print(result)

ApplyResult(registered_groups=['listing_features', 'town_market_features'], published=False)


In [5]:
store.list_feature_groups()

[FeatureGroupSummary(name='listing_features', owner='data-science-team', description='Historical sold listing attributes and prices', entity_key='listing_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=4),
 FeatureGroupSummary(name='town_market_features', owner='data-science-team', description='Monthly town-level market aggregate', entity_key='town_id', storage_target=<StorageTarget.OFFLINE_AND_ONLINE: 'OFFLINE_AND_ONLINE'>, feature_count=1)]

In [7]:
!kitefs list --format json

[
  {
    "name": "listing_features",
    "owner": "data-science-team",
    "description": "Historical sold listing attributes and prices",
    "entity_key": "listing_id",
    "storage_target": "OFFLINE",
    "feature_count": 4
  },
  {
    "name": "town_market_features",
    "owner": "data-science-team",
    "description": "Monthly town-level market aggregate",
    "entity_key": "town_id",
    "storage_target": "OFFLINE_AND_ONLINE",
    "feature_count": 1
  }
]


In [ ]:
!kitefs describe listing_features --format json

In [5]:
import pandas as pd

In [7]:
listing_features_df = pd.read_parquet("data/listing_features.parquet")

In [12]:
# listing_features_df.head()
# listing_features_df.describe()
listing_features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   listing_id       10000 non-null  int64              
 1   town_id          10000 non-null  int64              
 2   net_area         10000 non-null  int64              
 3   number_of_rooms  10000 non-null  int64              
 4   build_year       10000 non-null  int64              
 5   sold_price       10000 non-null  float64            
 6   sold_at          10000 non-null  datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(5)
memory usage: 547.0 KB


In [13]:
store.ingest("listing_features", listing_features_df)
store.ingest("town_market_features", "data/town_market_features.parquet")

IngestResult(feature_group='town_market_features', accepted_rows=72, rejected_rows=0, written_files=['/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=02/ing_20260606T055957_605917.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=03/ing_20260606T055957_a86820.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=04/ing_20260606T055957_517d5f.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=05/ing_20260606T055957_669141.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=06/ing_20260606T055957_e17cfc.parq

In [14]:
from datetime import datetime

listings = store.get_historical_features(
    from_="listing_features",
    select=["net_area", "number_of_rooms", "build_year", "sold_price"],
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 5, 1, 23, 59, 59),
        }
    },
)

In [17]:
listings

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price
0,1003,2025-04-15 17:17:51,5,139,3,2014,2359839.54
1,1020,2025-04-16 09:48:34,3,101,2,1985,2068432.36
2,1038,2025-04-23 12:39:51,1,105,3,2008,3410663.21
3,1041,2025-04-09 08:05:40,3,149,3,1996,3330769.37
4,1058,2025-04-10 16:42:40,6,145,3,1990,1974060.03
...,...,...,...,...,...,...,...
829,10180,2025-05-01 13:45:56,6,184,4,2000,2272088.95
830,10762,2025-05-01 19:48:20,4,65,2,2018,1311483.78
831,10946,2025-05-01 11:29:21,2,155,4,2020,5347999.65
832,10955,2025-05-01 14:06:56,6,143,3,1993,1834996.06


In [19]:
materialize_result = store.materialize("town_market_features")
print(materialize_result)

MaterializeResult(succeeded=['town_market_features'], skipped=[], failed=[])


In [20]:
import sqlite3
import pandas as pd

# Connect to your local db
conn = sqlite3.connect("feature_store/data/online_store/online.db")

# You can even use pandas to run the query and print a beautiful table!
df = pd.read_sql_query("select * from town_market_features;", conn)
print(df)

conn.close()

   town_id              event_timestamp  avg_price_per_sqm
0        1  2026-01-01T00:00:00.000000Z           31828.45
1        2  2026-01-01T00:00:00.000000Z           33041.63
2        3  2026-01-01T00:00:00.000000Z           23190.73
3        4  2026-01-01T00:00:00.000000Z           22122.53
4        5  2026-01-01T00:00:00.000000Z           17990.52
5        6  2026-01-01T00:00:00.000000Z           13610.89


In [22]:
market_price = store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

print(market_price)

{'town_id': 1, 'event_timestamp': datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), 'avg_price_per_sqm': 31828.45}


## Preparing Data and Model Training

In [23]:
from datetime import datetime

training_df = store.get_historical_features(
    from_="listing_features",
    join=["town_market_features"],
    select={
        "listing_features": [
            "net_area",
            "number_of_rooms",
            "build_year",
            "sold_price",
        ],
        "town_market_features": ["avg_price_per_sqm"],
    },
    where={
        "sold_at": {
            "gte": datetime(2025, 2, 1),
            "lte": datetime(2025, 12, 31, 23, 59, 59),
        }
    },
)

training_df

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
0,1001,2025-02-24 09:43:47,6,48,1,1995,592403.98,6,2025-02-01,12987.48
1,1002,2025-02-08 16:38:01,5,97,3,1981,1522575.69,5,2025-02-01,17093.62
2,1005,2025-02-02 19:29:34,4,117,3,2018,2352991.98,4,2025-02-01,21044.18
3,1007,2025-02-28 11:55:06,6,47,1,1998,673645.54,6,2025-02-01,12987.48
4,1029,2025-02-25 11:10:26,5,99,2,1980,1850068.46,5,2025-02-01,17093.62
...,...,...,...,...,...,...,...,...,...,...
9139,10930,2025-12-08 16:06:48,1,114,3,2007,3424869.72,1,2025-12-01,31540.58
9140,10944,2025-12-22 18:40:30,3,161,4,2016,3676213.34,3,2025-12-01,23037.48
9141,10966,2025-12-25 13:19:35,4,175,4,1984,4000532.01,4,2025-12-01,22134.49
9142,10971,2025-12-27 16:46:58,6,149,3,2018,2026020.75,6,2025-12-01,13660.93


In [24]:
FEATURE_COLUMNS = [
    "net_area",
    "number_of_rooms",
    "build_year",
    "town_market_features_avg_price_per_sqm",
]
LABEL_COLUMN = "sold_price"

X = training_df[FEATURE_COLUMNS]
y = training_df[LABEL_COLUMN]

In [26]:
print(X.head())

print(y.head())

   net_area  number_of_rooms  build_year  \
0        48                1        1995   
1        97                3        1981   
2       117                3        2018   
3        47                1        1998   
4        99                2        1980   

   town_market_features_avg_price_per_sqm  
0                                12987.48  
1                                17093.62  
2                                21044.18  
3                                12987.48  
4                                17093.62  
0     592403.98
1    1522575.69
2    2352991.98
3     673645.54
4    1850068.46
Name: sold_price, dtype: float64


In [27]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [28]:
from sklearn.metrics import mean_absolute_error, r2_score

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R2 :", r2_score(y_test, predictions))

MAE: 153651.73382779333
R2 : 0.9777948434747418


In [29]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)
joblib.dump(model, "models/model.pkl")

['models/model.pkl']

In [ ]:
import os

# 1. Select AWS credential profile
os.environ["AWS_PROFILE"] = "aws-credential-profile"

# 2. Tell KiteFS to target the remote AWS provider
os.environ["KITEFS_RUNTIME_TARGET"] = "remote"
os.environ["KITEFS_AWS_REGION"] = "us-east-1"

# 3. Tell KiteFS to use your new bucket
os.environ["KITEFS_REMOTE_REGISTRY_S3_BUCKET"] = "s3-bucket-name"
os.environ["KITEFS_REMOTE_REGISTRY_S3_PREFIX"] = "kitefs" 

os.environ["KITEFS_REMOTE_OFFLINE_S3_BUCKET"] = "s3-bucket-name"
os.environ["KITEFS_REMOTE_OFFLINE_S3_PREFIX"] = "kitefs"

# 4. Set the DynamoDB prefix
os.environ["KITEFS_REMOTE_ONLINE_DYNAMODB_TABLE_PREFIX"] = "kitefs_"

print("AWS environment variables loaded successfully!")

AWS environment variables loaded successfully!


In [4]:
# Build a fresh, AWS-bound store instance
remote_store = FeatureStore()

In [7]:
apply_result = remote_store.apply(publish=True)
print("Registered groups:", apply_result.registered_groups)
print("Published successfully:", apply_result.published)

Registered groups: ['listing_features', 'town_market_features']
Published successfully: True


In [8]:
print("Ingesting listing features to S3...")
remote_store.ingest("listing_features", "data/listing_features.parquet")

print("Ingesting town market aggregates to S3...")
remote_store.ingest("town_market_features", "data/town_market_features.parquet")

print("Ingestion complete!")

Ingesting listing features to S3...
Ingesting town market aggregates to S3...
Ingestion complete!


In [9]:
remote_store.list_feature_groups()

[FeatureGroupSummary(name='listing_features', owner='data-science-team', description='Historical sold listing attributes and prices', entity_key='listing_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=4),
 FeatureGroupSummary(name='town_market_features', owner='data-science-team', description='Monthly town-level market aggregate', entity_key='town_id', storage_target=<StorageTarget.OFFLINE_AND_ONLINE: 'OFFLINE_AND_ONLINE'>, feature_count=1)]

In [10]:
materialize_result = remote_store.materialize("town_market_features")
print(materialize_result)


MaterializeResult(succeeded=['town_market_features'], skipped=[], failed=[])


## Retraining

In [11]:
!kitefs ingest listing_features data/listing_features_jan2026.parquet

Ingested 1000 row(s) into 'listing_features'. Rejected 0 row(s). Wrote 1 file(s).
  Validation: 1000 passed, 0 failed.


In [12]:
!kitefs ingest town_market_features data/town_market_jan2026.parquet

Ingested 6 row(s) into 'town_market_features'. Rejected 0 row(s). Wrote 1 file(s).
  Validation: 6 passed, 0 failed.


In [13]:
!kitefs materialize town_market_features

Succeeded: town_market_features. Skipped: (none). Failed: (none).


In [14]:
market_price = remote_store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

print(market_price)

{'town_id': 1, 'event_timestamp': datetime.datetime(2026, 2, 1, 0, 0, tzinfo=datetime.timezone.utc), 'avg_price_per_sqm': 31346.96}


In [16]:
from datetime import datetime

training_df = remote_store.get_historical_features(
    from_="listing_features",
    join=["town_market_features"],
    select={
        "listing_features": [
            "net_area",
            "number_of_rooms",
            "build_year",
            "sold_price",
        ],
        "town_market_features": ["avg_price_per_sqm"],
    },
    where={
        "sold_at": {
            "gte": datetime(2025, 2, 1),
            "lte": datetime(2026, 1, 31, 23, 59, 59),
        }
    },
)

In [17]:
training_df.describe()

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
count,10144.000000,10144,10144.000000,10144.000000,10144.000000,10144.000000,1.014400e+04,10144.000000,10144,10144.000000
mean,7421.676065,2025-08-02 10:44:00.307373,3.517646,121.891759,2.900039,2002.730678,2.821108e+06,3.517646,2025-07-18 06:01:42.208202,23050.265600
min,1001.000000,2025-02-01 08:01:04,1.000000,40.000000,1.000000,1980.000000,5.029452e+05,1.000000,2025-02-01 00:00:00,12987.480000
25%,3749.750000,2025-05-02 19:24:51,2.000000,92.000000,2.000000,1991.000000,1.828113e+06,2.000000,2025-05-01 00:00:00,17533.310000
50%,6535.500000,2025-08-01 12:33:12.500000,4.000000,115.000000,3.000000,2003.000000,2.562554e+06,4.000000,2025-08-01 00:00:00,22122.530000
75%,9310.250000,2025-11-02 18:27:20,5.000000,147.000000,3.000000,2014.000000,3.515269e+06,5.000000,2025-11-01 00:00:00,31077.360000
max,21000.000000,2026-01-31 19:33:43,6.000000,250.000000,5.000000,2025.000000,8.902179e+06,6.000000,2026-01-01 00:00:00,33230.620000
std,5122.419154,NaN,1.703637,41.963004,0.944514,13.239697,1.332929e+06,1.703637,NaN,6841.605884


In [18]:
training_df.head()

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
0,1001,2025-02-24 09:43:47,6,48,1,1995,592403.98,6,2025-02-01,12987.48
1,1002,2025-02-08 16:38:01,5,97,3,1981,1522575.69,5,2025-02-01,17093.62
2,1005,2025-02-02 19:29:34,4,117,3,2018,2352991.98,4,2025-02-01,21044.18
3,1007,2025-02-28 11:55:06,6,47,1,1998,673645.54,6,2025-02-01,12987.48
4,1029,2025-02-25 11:10:26,5,99,2,1980,1850068.46,5,2025-02-01,17093.62


In [19]:
FEATURE_COLUMNS = [
    "net_area",
    "number_of_rooms",
    "build_year",
    "town_market_features_avg_price_per_sqm",
]
LABEL_COLUMN = "sold_price"

X = training_df[FEATURE_COLUMNS]
y = training_df[LABEL_COLUMN]

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, r2_score

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R2 :", r2_score(y_test, predictions))

import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)
joblib.dump(model, "models/model.pkl")

MAE: 151370.2854069356
R2 : 0.9785126010713899


['models/model.pkl']